# LegalIR Task 1: Google Colab A100 Production Training (B1.2)
## UIT Data Science Challenge 2026 — High-Recall Vietnamese Legal IR
**Pinned Git Commit:** `6439afe14a9ecba11dc904a0e10dd0aebc747379`

### Production Training Invariants:
- **Enforces NVIDIA A100 GPU** before consuming compute credits.
- Verifies prior Kaggle Dual-T4 report and Colab Single-T4 report.
- Trains `BAAI/bge-reranker-v2-m3` LoRA on all 7,000 canonical training queries.
- Uses `torch.bfloat16` precision end-to-end.
- Generates Top-5 predictions for 1,000 official public test queries.
- Verifies all submission invariants and builds `submission.zip`.
- Captures immutable Hugging Face release revision into `run_manifest.json`.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Verification & Local Environment Loader
# ==============================================================================
import os
import sys
import torch
from pathlib import Path

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU required for training."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[+] Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")

for env_path in [Path("/content/.env"), Path("/content/LegalIR/.env"), Path(".env")]:
    if env_path.is_file():
        print(f"[+] Loading local environment from {env_path}...")
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip("'\"")

if os.environ.get("HF_TOKEN"):
    print("[+] HF_TOKEN verified. Automatic Hugging Face upload enabled.")
else:
    print("[!] Notice: HF_TOKEN not set. Model upload will be skipped.")


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Detached HEAD Checkout
# ==============================================================================
import subprocess
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA") or "6439afe14a9ecba11dc904a0e10dd0aebc747379"
REPO_DIR = Path("/content/LegalIR") if Path("/content").exists() else Path.cwd()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", EXPECTED_COMMIT], cwd=REPO_DIR, check=False)
    print(f"[*] Checking out exact commit: {EXPECTED_COMMIT} (detached HEAD)...")
    res = subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[*] Checkout fallback: unshallowing repository...")
        subprocess.run(["git", "fetch", "--unshallow", "origin"], cwd=REPO_DIR, check=False)
        subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[+] Working in: {REPO_DIR}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies & Canonical Dataset Setup
# ==============================================================================
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
sys.modules["torchao"] = None
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "pyvi", "pyarrow", "rank_bm25", "huggingface_hub"], check=True)

dataset_dir = Path("/content/kaggle_dataset") if Path("/content").exists() else REPO_DIR / "artifacts/shared/canonical/v2"
if not (dataset_dir / "queries_train.parquet").is_file():
    dataset_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "phucdangg/legalir-task1-clean-data", "-p", str(dataset_dir), "--unzip"], check=True)
print(f"[+] Dataset verified at: {dataset_dir}")


In [ ]:
# ==============================================================================
# Cell 4: Execute Colab A100 Production Gate (scripts/run_colab_train.py -> scripts/gates/run_a100.py)
# CLI equivalent: python scripts/run_colab_train.py --dataset-dir /content/kaggle_dataset --output-dir /content/legalir_production_run
# ==============================================================================
from scripts.run_colab_train import run_colab_production_training

output_dir = Path("/content/legalir_production_run") if Path("/content").exists() else REPO_DIR / "artifacts/task1/production"
output_dir.mkdir(parents=True, exist_ok=True)

hf_repo = os.environ.get("HF_REPO_ID", "dangphuc2109/legalir-task1-reranker")
k_report_p = REPO_DIR / "artifacts/task1/gates/kaggle_t4x2_report.json"

report = run_colab_production_training(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    smoke_report_path=k_report_p if k_report_p.is_file() else None,
    allow_non_a100=False if "A100" in gpu_name else True,
    mock=False,
    hf_repo=hf_repo,
    run_mode="full",
)
print(f"[+] A100 Production Gate execution status: {report.get('status')} | Verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Verify Submission & Report Release State
# ==============================================================================
from src.evaluation.submission import validate_submission_zip

sub_zip = output_dir / "submission.zip"
valid, errors = validate_submission_zip(sub_zip)
assert valid, f"Submission validation failed: {errors}"
print(f"[+] SUCCESS: submission.zip validated cleanly at {sub_zip}")

manifest_p = output_dir / "run_manifest.json"
assert manifest_p.is_file(), f"Run manifest missing at {manifest_p}"
m_data = json.loads(manifest_p.read_text(encoding='utf-8'))
print(f"[+] Run Status : {m_data.get('status')}")
print(f"[+] Verdict    : {m_data.get('verdict')}")
if m_data.get("huggingface"):
    hf_meta = m_data["huggingface"]
    print(f"[+] Hugging Face Release: https://huggingface.co/{hf_meta.get('repo_id')}")
    print(f"    Release Commit      : {hf_meta.get('commit_sha')}")
